# WEEK 2 DAY 5 EXERCISE: AI CHATBOT ASSISTANT FOR BUYING FOOTBALL TICKETS. 




### I created an AI chat assistant for bying West Ham tickets specifically for four upcoming home matches (2026). Customers can choose between different ticket packages. The VIP option is clearly explained so that customers understand why the VIP package is priced higher. I also customised the gradio interface to match with West Ham's colours.

### **Implemented with**: OpenAI API calls, Gradio UI, tool calling, database integration, and multimodal features (image and sound)

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()



In [ ]:
system_message = """You are an assistant that helps WEST-HAM football 
customers buying tickets for both FA CUP and Premier League matches. Ask what matches date they want.
After the customer replies with the date, confirm the date and the match and ask the option to buy only the basic ticket
which include the seat at the stadium or 'the all inclusive option- which includes buffet and access to the VIP lounge'. 
Here explain a bit more the advantage of getting VIP access (e.g. fine dining buffet, top floor lounge with former legends inside)
You shoud provide price and option of how many tickets to buy. After that, you must ask for their names before completing the booking.  
Once the user confirm the names, confirm the booking (make sure to cite either the vip or basic ticket) and tell them to go to https://www.whufc.com/en to print the ticket or simply scan the qr code to access it. 
When talking try to use a friendly informal football fun tone. Like a West Ham supporter speaking to another fan. keep it relax and enthusiastic but still clear and helpful. 
DO not claim that you cannot generate images. the ticket image will be created separately by the application. 
do not skip the step of asking for ticket names.
If the user gives a date like '5 April', 'April 5', or '5th April', 
treat it as a 2026 date (2026-04-05) and use the tool. same for the 10th of april, treat it as 2026-04-10, 25 april and 10 may matches.
Do not assume 2024 or any other year.
"""

In [ ]:
DB = "tickets_football.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""CREATE TABLE IF NOT EXISTS matches (date TEXT PRIMARY KEY, 
    home TEXT,
    away TEXT,
    location TEXT,
    basic_price REAL,
    all_inclusive_price REAL
    )""")
    conn.commit()
     
matches = { 
    "2026-04-05": {
    "home": "West Ham",
    "away": "Leeds United",
    "location": "London Stadium",
    "basic_price": 95,
    "all_inclusive_price": 230
 },

 "2026-04-10": {
    "home": "West Ham",
    "away": "Wolves",
    "location": "London Stadium",
    "basic_price": 56,
    "all_inclusive_price": 245
 },

 "2026-04-25": {
    "home": "West Ham",
    "away": "Everton",
    "location": "London Stadium",
    "basic_price": 41,
    "all_inclusive_price": 215
 },

 
 "2026-05-10": {
    "home": "West Ham",
    "away": "Arsenal",
    "location": "London Stadium",
    "basic_price": 105,
    "all_inclusive_price": 280
 },

}




def set_matches_info(date, match_info):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute( """INSERT or REPLACE INTO matches 
        (date, home, away, location, basic_price, all_inclusive_price) VALUES (?, ?, ?, ?, ?, ?) """, (date, 
        match_info["home"],
        match_info["away"],
        match_info["location"],
        match_info["basic_price"],
        match_info["all_inclusive_price"]))
        conn.commit()    

for date, match_info in matches.items():
    set_matches_info(date, match_info)  


In [ ]:
def get_match_info(date):
    print(f"DATABASE TOOL CALLED: getting match info for {date}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT home, away, location, basic_price, all_inclusive_price FROM matches WHERE date = ?",
            (date,)
        )
        result = cursor.fetchone()

    if result:
        return (
            f"Match on {date}: {result[0]} vs {result[1]} at {result[2]}. "
            f"Basic ticket: £{result[3]}, all-inclusive ticket: £{result[4]}."
        )
    else:
        return f"No match data available for {date}"

In [ ]:
match_info_function = {
    "name": "get_match_info",
    "description": "Get the football match information for a given date, including teams, location, and ticket prices.",
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "The date of the match in YYYY-MM-DD format."
            }
        },
        "required": ["date"]
    }
}

tools = [{"type": "function", "function": match_info_function}]
tools

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:

def artist(ticket_date, holder_name, ticket_type):
    prompt = f"""
    Create a realistic premium football match ticket for West Ham United.

    Ticket holder: {holder_name}
    
    Date: {ticket_date}
    Location: London Stadium
    Ticket type: {ticket_type}
    includes seat stadium

    Make it look like a real printed match ticket, professional, elegant, high quality,
    with a barcode area, a unique qr code and club-style design. Do not add West ham vs the other team. just "West Ham United"
    """

    try:
        image_response = openai.images.generate(
            model="gpt-image-1",
            prompt=prompt,
            size="1024x1024"
        )

        if not image_response.data:
            raise ValueError("No image data returned.")

        if not hasattr(image_response.data[0], "b64_json") or image_response.data[0].b64_json is None:
            raise ValueError("Image response did not contain b64_json.")

        image_base64 = image_response.data[0].b64_json
        image_data = base64.b64decode(image_base64)

        return Image.open(BytesIO(image_data))

    except Exception as e:
        print("ARTIST ERROR:", repr(e))
        return None

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="coral",    
      input=message
    )
    return response.content

In [ ]:
def chat(history, booking_state):
    history = history or []
    booking_state = booking_state or {
        "ticket_date": None,
        "awaiting_name": False,
        "holder_name": None
    }

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    last_user_message = None
    for item in reversed(history):
        if item["role"] == "user":
            last_user_message = item["content"]
            break

    image = None

    print("DEBUG booking_state start:", booking_state)
    print("DEBUG last_user_message:", last_user_message)

   #assistant is waiting for the user's name
    if booking_state["awaiting_name"] and booking_state["ticket_date"] and last_user_message:
        holder_name = last_user_message.strip()

        image = artist(
        booking_state["ticket_date"],
        holder_name,
        booking_state.get("ticket_type", "all-inclusive")
            )

        if image is not None:
            reply = (
                f"Cheers, {holder_name}! Your ticket for the FA Cup match on "
                f"{booking_state['ticket_date']} is booked.\n\n"
                f"You can head over to https://www.whufc.com/en to print your ticket or simply scan the qr code with your phone for direct download."
            )
        else:
            reply = (
                f"Cheers, {holder_name}! Your booking is complete, but I hit a problem while "
                f"creating the ticket image.\n\n"
                f"You can still head over to https://www.whufc.com/en to print your ticket  ."
            )

        history += [{"role": "assistant", "content": reply}]

        try:
            voice = talker(reply)
        except Exception as e:
            print("TALKER ERROR:", repr(e))
            voice = None

        booking_state["awaiting_name"] = False
        booking_state["holder_name"] = holder_name

        print("DEBUG booking_state end case 1:", booking_state)
        return history, voice, image, booking_state

    # normal assistant flow
    messages = [{"role": "system", "content": system_message}] + history

    try:
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
    except Exception as e:
        print("CHAT API ERROR:", repr(e))
        reply = f"I hit an error while processing the booking: {e}"
        history += [{"role": "assistant", "content": reply}]
        try:
            voice = talker(reply)
        except Exception:
            voice = None
        return history, voice, None, booking_state

    tickets = []

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, tickets = handle_tool_calls_and_return_dates(message)

        print("DEBUG tickets from tool:", tickets)

        if tickets:
            booking_state["ticket_date"] = tickets[0]

        messages.append(message)
        messages.extend(responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content
    reply_lower = reply.lower()

    if "all-inclusive" in reply_lower:
        booking_state["ticket_type"] = "all-inclusive"
    elif "basic" in reply_lower:
        booking_state["ticket_type"] = "basic"
    history += [{"role": "assistant", "content": reply}]

    try:
        voice = talker(reply)
    except Exception as e:
        print("TALKER ERROR:", repr(e))
        voice = None

    if "name" in reply.lower():
        booking_state["awaiting_name"] = True

    print("DEBUG booking_state end case 2:", booking_state)
    return history, voice, image, booking_state

         

In [ ]:
def handle_tool_calls_and_return_dates(message):
    responses = []
    tickets = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_match_info":
            arguments = json.loads(tool_call.function.arguments)
            raw_date = arguments.get("date")
            date = normalize_date(raw_date)

            if date:
                tickets.append(date)
                match_details = get_match_info(date)
            else:
                match_details = "Sorry, I couldn't understand the date. Please use something like '5 April'."

            responses.append({
                "role": "tool",
                "content": match_details,
                "tool_call_id": tool_call.id
            })

    return responses, tickets

In [ ]:
from datetime import datetime

def normalize_date(user_input):
    if not user_input:
        return None

    cleaned = (
        user_input.lower()
        .replace("st", "")
        .replace("nd", "")
        .replace("rd", "")
        .replace("th", "")
        .strip()
    )

    for fmt in ["%d %B", "%B %d", "%Y-%m-%d"]:
        try:
            dt = datetime.strptime(cleaned, fmt)
            if fmt != "%Y-%m-%d":
                dt = dt.replace(year=2026)
            return dt.strftime("%Y-%m-%d")
        except:
            continue

    return None

In [ ]:
import gradio as gr

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

initial_booking_state = {
    "ticket_date": None,
    "awaiting_name": False,
    "holder_name": None,
    "ticket_type": None   
}

custom_theme = gr.themes.Soft()

css = """
:root {
    --whu-claret: #7A263A;
    --whu-claret-dark: #5f1d2d;
    --whu-claret-soft: rgba(122, 38, 58, 0.08);
    --whu-claret-softer: rgba(122, 38, 58, 0.04);
    --whu-border: rgba(122, 38, 58, 0.18);
}

/* overall page */
.gradio-container {
    background:
        linear-gradient(135deg, var(--whu-claret-soft), #ffffff 45%, var(--whu-claret-softer));
}

/* main title block if you add one later */
#app_shell {
    border-radius: 18px;
}

/* chatbot panel */
#wh_chatbot {
    border: 1px solid var(--whu-border) !important;
    border-radius: 16px !important;
    overflow: hidden;
    background: rgba(255, 255, 255, 0.88) !important;
    backdrop-filter: blur(6px);
}

/* image panel */
#ticket_image {
    border: 1px solid var(--whu-border) !important;
    border-radius: 16px !important;
    overflow: hidden;
    background: rgba(255, 255, 255, 0.9) !important;
}

/* audio panel */
#audio_panel {
    border: 1px solid var(--whu-border) !important;
    border-radius: 14px !important;
    background: rgba(255, 255, 255, 0.82) !important;
}

/* textbox wrapper */
#message_box textarea,
#message_box input {
    border: 1px solid var(--whu-border) !important;
    border-radius: 14px !important;
    background: rgba(255, 255, 255, 0.95) !important;
}

/* submit button inside textbox row if present */
button.primary {
    background: var(--whu-claret) !important;
    border: none !important;
    color: white !important;
}

button.primary:hover {
    background: var(--whu-claret-dark) !important;
}

/* generic component containers */
#chat_col > div,
#image_col > div {
    border-radius: 16px !important;
}

/* optional subtle headings/labels */
label, .gradio-container h1, .gradio-container h2, .gradio-container h3 {
    color: var(--whu-claret);
}
"""

with gr.Blocks(theme=custom_theme, css=css, elem_id="app_shell") as ui:
    booking_state = gr.State(initial_booking_state)

    gr.Markdown("## West Ham Ticket Assistant")

    with gr.Row():
        with gr.Column(elem_id="chat_col"):
            chatbot = gr.Chatbot(
                height=500,
                type="messages",
                elem_id="wh_chatbot"
            )

        with gr.Column(elem_id="image_col"):
            image_output = gr.Image(
                height=500,
                interactive=False,
                elem_id="ticket_image"
            )

    with gr.Row():
        audio_output = gr.Audio(
            autoplay=True,
            elem_id="audio_panel"
        )

    with gr.Row():
        message = gr.Textbox(
            label="Chat with our AI Assistant:",
            placeholder="Ask about the next match or ticket options...",
            elem_id="message_box"
        )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        chat,
        inputs=[chatbot, booking_state],
        outputs=[chatbot, audio_output, image_output, booking_state]
    )

ui.launch(inbrowser=True)